In [2]:
import pandas as pd
from pathlib import Path

In [1]:
import pandas as pd
from pathlib import Path

# File path
file_path = Path("../data/processed/purchases_features.csv")

# Load dataset
df = pd.read_csv(file_path)

# Basic verification
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nDate range:")
print(df["date"].min(), "→", df["date"].max())

print("\nMissing values:", df.isnull().sum().sum())

print("\nDuplicate date + product_id rows:",
      df.duplicated(subset=["date", "product_id"]).sum())

Shape: (247902, 27)

Columns:
['date', 'product_id', 'demand', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'day_of_year', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_7', 'expanding_mean', 'expanding_std', 'previous_count', 'product_total_demand', 'product_avg_demand', 'previous_price', 'price_change', 'product_unique_customers']

Date range:
2014-01-01 → 2015-12-30

Missing values: 0

Duplicate date + product_id rows: 0


In [3]:
# 7.2 — Check date ordering and available dates

# Convert date to datetime
df["date"] = pd.to_datetime(df["date"])

# Sort chronologically
df = df.sort_values("date").reset_index(drop=True)

# Unique dates
unique_dates = df["date"].drop_duplicates()

print("First date:", unique_dates.min())
print("Last date:", unique_dates.max())

print("\nTotal unique dates:", len(unique_dates))

print("\nFirst 10 dates:")
print(unique_dates.head(10).to_list())

print("\nLast 10 dates:")
print(unique_dates.tail(10).to_list())

# Check whether dates are in chronological order
is_sorted = df["date"].is_monotonic_increasing

print("\nDate column sorted:", is_sorted)

First date: 2014-01-01 00:00:00
Last date: 2015-12-30 00:00:00

Total unique dates: 728

First 10 dates:
[Timestamp('2014-01-01 00:00:00'), Timestamp('2014-01-02 00:00:00'), Timestamp('2014-01-03 00:00:00'), Timestamp('2014-01-04 00:00:00'), Timestamp('2014-01-05 00:00:00'), Timestamp('2014-01-06 00:00:00'), Timestamp('2014-01-07 00:00:00'), Timestamp('2014-01-08 00:00:00'), Timestamp('2014-01-09 00:00:00'), Timestamp('2014-01-10 00:00:00')]

Last 10 dates:
[Timestamp('2015-12-21 00:00:00'), Timestamp('2015-12-22 00:00:00'), Timestamp('2015-12-23 00:00:00'), Timestamp('2015-12-24 00:00:00'), Timestamp('2015-12-25 00:00:00'), Timestamp('2015-12-26 00:00:00'), Timestamp('2015-12-27 00:00:00'), Timestamp('2015-12-28 00:00:00'), Timestamp('2015-12-29 00:00:00'), Timestamp('2015-12-30 00:00:00')]

Date column sorted: True


In [4]:
# 7.3 — Define chronological split dates

unique_dates = df["date"].drop_duplicates().sort_values().reset_index(drop=True)

n_dates = len(unique_dates)

# 70% train, 15% validation, 15% test
train_end_idx = int(n_dates * 0.70) - 1
validation_end_idx = int(n_dates * 0.85) - 1

train_end_date = unique_dates.iloc[train_end_idx]
validation_end_date = unique_dates.iloc[validation_end_idx]

print("Total unique dates:", n_dates)

print("\nTrain:")
print("2014-01-01 →", train_end_date)

print("\nValidation:")
print(train_end_date + pd.Timedelta(days=1), "→", validation_end_date)

print("\nTest:")
print(validation_end_date + pd.Timedelta(days=1), "→", unique_dates.iloc[-1])

Total unique dates: 728

Train:
2014-01-01 → 2015-05-25 00:00:00

Validation:
2015-05-26 00:00:00 → 2015-09-11 00:00:00

Test:
2015-09-12 00:00:00 → 2015-12-30 00:00:00


C:\Users\rar95\AppData\Local\Temp\ipykernel_12872\3907840526.py:20: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  print(train_end_date + pd.Timedelta(days=1), "→", validation_end_date)
C:\Users\rar95\AppData\Local\Temp\ipykernel_12872\3907840526.py:23: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`).Please use a specific unit instead.
  print(validation_end_date + pd.Timedelta(days=1), "→", unique_dates.iloc[-1])


In [5]:
# 7.4 — Create chronological train/validation/test datasets

# Define exact split boundaries
train_end_date = pd.Timestamp("2015-05-25")
validation_end_date = pd.Timestamp("2015-09-11")

# Create chronological splits
train_df = df[df["date"] <= train_end_date].copy()

validation_df = df[
    (df["date"] > train_end_date) &
    (df["date"] <= validation_end_date)
].copy()

test_df = df[df["date"] > validation_end_date].copy()

# Display results
print("TRAIN")
print("Rows:", len(train_df))
print("Date:", train_df["date"].min(), "→", train_df["date"].max())

print("\nVALIDATION")
print("Rows:", len(validation_df))
print("Date:", validation_df["date"].min(), "→", validation_df["date"].max())

print("\nTEST")
print("Rows:", len(test_df))
print("Date:", test_df["date"].min(), "→", test_df["date"].max())

print("\nTOTAL SPLIT ROWS:", len(train_df) + len(validation_df) + len(test_df))
print("ORIGINAL ROWS:", len(df))

TRAIN
Rows: 103272
Date: 2014-01-01 00:00:00 → 2015-05-25 00:00:00

VALIDATION
Rows: 63109
Date: 2015-05-26 00:00:00 → 2015-09-11 00:00:00

TEST
Rows: 81521
Date: 2015-09-12 00:00:00 → 2015-12-30 00:00:00

TOTAL SPLIT ROWS: 247902
ORIGINAL ROWS: 247902


In [6]:
# 7.5 — Verify no date overlap between splits

train_max = train_df["date"].max()
validation_min = validation_df["date"].min()
validation_max = validation_df["date"].max()
test_min = test_df["date"].min()

print("Train max date:", train_max)
print("Validation min date:", validation_min)

print("\nValidation max date:", validation_max)
print("Test min date:", test_min)

print("\nTrain → Validation separation:",
      train_max < validation_min)

print("Validation → Test separation:",
      validation_max < test_min)

# Check for actual overlapping dates
train_dates = set(train_df["date"])
validation_dates = set(validation_df["date"])
test_dates = set(test_df["date"])

print("\nTrain ∩ Validation:", len(train_dates & validation_dates))
print("Validation ∩ Test:", len(validation_dates & test_dates))
print("Train ∩ Test:", len(train_dates & test_dates))

Train max date: 2015-05-25 00:00:00
Validation min date: 2015-05-26 00:00:00

Validation max date: 2015-09-11 00:00:00
Test min date: 2015-09-12 00:00:00

Train → Validation separation: True
Validation → Test separation: True

Train ∩ Validation: 0
Validation ∩ Test: 0
Train ∩ Test: 0


In [7]:
# 7.6 — Verify chronological ordering and row counts

print("TRAIN")
print("Chronologically sorted:", train_df["date"].is_monotonic_increasing)
print("Rows:", len(train_df))

print("\nVALIDATION")
print("Chronologically sorted:", validation_df["date"].is_monotonic_increasing)
print("Rows:", len(validation_df))

print("\nTEST")
print("Chronologically sorted:", test_df["date"].is_monotonic_increasing)
print("Rows:", len(test_df))

# Final row-count verification
total_split_rows = len(train_df) + len(validation_df) + len(test_df)

print("\nTotal split rows:", total_split_rows)
print("Original rows:", len(df))
print("Row count matches:", total_split_rows == len(df))

TRAIN
Chronologically sorted: True
Rows: 103272

VALIDATION
Chronologically sorted: True
Rows: 63109

TEST
Chronologically sorted: True
Rows: 81521

Total split rows: 247902
Original rows: 247902
Row count matches: True


In [8]:
# 7.7 — Compare demand distribution across splits

print("TRAIN DEMAND")
print(train_df["demand"].describe())

print("\nVALIDATION DEMAND")
print(validation_df["demand"].describe())

print("\nTEST DEMAND")
print(test_df["demand"].describe())

TRAIN DEMAND
count    103272.000000
mean         18.223671
std          59.827183
min           1.000000
25%           2.000000
50%           6.000000
75%          15.000000
max        3906.000000
Name: demand, dtype: float64

VALIDATION DEMAND
count    63109.000000
mean        21.341108
std         58.889938
min          1.000000
25%          3.000000
50%          8.000000
75%         22.000000
max       4300.000000
Name: demand, dtype: float64

TEST DEMAND
count    81521.000000
mean        23.028275
std         77.149495
min          1.000000
25%          3.000000
50%          8.000000
75%         24.000000
max      12540.000000
Name: demand, dtype: float64


In [9]:
# 7.8 — Verify features and missing values

splits = {
    "TRAIN": train_df,
    "VALIDATION": validation_df,
    "TEST": test_df
}

for name, data in splits.items():
    print(f"\n{name}")
    print("-" * len(name))

    print("Shape:", data.shape)
    print("Missing values:", data.isnull().sum().sum())
    print("Target present:", "demand" in data.columns)

    missing_columns = [
        col for col in df.columns
        if col not in data.columns
    ]

    print("Missing columns:", missing_columns)
    


TRAIN
-----
Shape: (103272, 27)
Missing values: 0
Target present: True
Missing columns: []

VALIDATION
----------
Shape: (63109, 27)
Missing values: 0
Target present: True
Missing columns: []

TEST
----
Shape: (81521, 27)
Missing values: 0
Target present: True
Missing columns: []


In [10]:
# 7.9 — Check features around split boundaries

feature_columns = [
    col for col in df.columns
    if col not in ["date", "product_id", "demand"]
]

print("Number of feature columns:", len(feature_columns))

print("\nFeature columns:")
print(feature_columns)

# Inspect the last training date
print("\nLast 3 training dates:")
print(
    train_df[["date", "product_id", "demand"] + feature_columns]
    .tail(3)
)

# Inspect the first validation date
print("\nFirst 3 validation dates:")
print(
    validation_df[["date", "product_id", "demand"] + feature_columns]
    .head(3)
)

# Inspect the first test date
print("\nFirst 3 test dates:")
print(
    test_df[["date", "product_id", "demand"] + feature_columns]
    .head(3)
)

Number of feature columns: 24

Feature columns:
['year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'day_of_year', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_30', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_30', 'rolling_std_7', 'expanding_mean', 'expanding_std', 'previous_count', 'product_total_demand', 'product_avg_demand', 'previous_price', 'price_change', 'product_unique_customers']

Last 3 training dates:
             date  product_id  demand  year  month  day  day_of_week  \
103269 2015-05-25        4028      36  2015      5   25            0   
103270 2015-05-25        1121       6  2015      5   25            0   
103271 2015-05-25        1665      20  2015      5   25            0   

        week_of_year  quarter  day_of_year  ...  rolling_mean_30  \
103269            22        2          145  ...        66.500000   
103270            22        2          145  ...        11.933333   
103271            22        2          145  ...        41.966667   

In [11]:
# 7.9 — Check potential leakage in historical aggregate features

historical_features = [
    "expanding_mean",
    "expanding_std",
    "previous_count",
    "product_total_demand",
    "product_avg_demand",
    "product_unique_customers"
]

print("Checking historical/aggregate features:\n")

for feature in historical_features:
    print(
        f"{feature}: "
        f"min={df[feature].min():.2f}, "
        f"max={df[feature].max():.2f}, "
        f"mean={df[feature].mean():.2f}"
    )

# Check whether aggregate features vary over time
print("\nUnique values by feature:")

for feature in historical_features:
    print(f"{feature}: {df[feature].nunique()}")

Checking historical/aggregate features:

expanding_mean: min=0.00, max=3264.00, mean=20.59
expanding_std: min=0.00, max=2605.25, mean=29.55
previous_count: min=0.00, max=696.00, mean=71.36
product_total_demand: min=0.00, max=54271.00, mean=1535.29
product_avg_demand: min=0.00, max=3264.00, mean=20.59
product_unique_customers: min=0.00, max=1783.00, mean=101.95

Unique values by feature:
expanding_mean: 105984
expanding_std: 218539
previous_count: 697
product_total_demand: 13032
product_avg_demand: 105984
product_unique_customers: 1513


In [12]:
# 7.9 — Inspect historical features for temporal leakage

check_cols = [
    "date",
    "product_id",
    "demand",
    "previous_count",
    "product_total_demand",
    "product_avg_demand",
    "product_unique_customers",
    "expanding_mean",
    "expanding_std"
]

# Show one product across time
sample_product = df["product_id"].value_counts().index[0]

product_history = (
    df[df["product_id"] == sample_product]
    [check_cols]
    .sort_values("date")
)

print("Sample product_id:", sample_product)
print("\nFirst 10 observations:")
print(product_history.head(10).to_string(index=False))

print("\nLast 10 observations:")
print(product_history.tail(10).to_string(index=False))

Sample product_id: 3891

First 10 observations:
      date  product_id  demand  previous_count  product_total_demand  product_avg_demand  product_unique_customers  expanding_mean  expanding_std
2014-01-01        3891       6               0                   0.0            0.000000                       0.0        0.000000       0.000000
2014-01-02        3891       9               1                   6.0            6.000000                       2.0        6.000000       0.000000
2014-01-03        3891       3               2                  15.0            7.500000                       6.0        7.500000       2.121320
2014-01-05        3891      10               3                  18.0            6.000000                       8.0        6.000000       3.000000
2014-01-06        3891       4               4                  28.0            7.000000                      13.0        7.000000       3.162278
2014-01-07        3891      25               5                  32.0        

### Final verification of saved split datasets

In [ ]:
# 7.10 — Save chronological train/validation/test datasets

from pathlib import Path

output_dir = Path("../data/processed")

train_path = output_dir / "train.csv"
validation_path = output_dir / "validation.csv"
test_path = output_dir / "test.csv"

train_df.to_csv(train_path, index=False)
validation_df.to_csv(validation_path, index=False)
test_df.to_csv(test_path, index=False)

print("Files saved successfully:\n")

print("Train:", train_path)
print("Validation:", validation_path)
print("Test:", test_path)

Files saved successfully:

Train: ..\data\processed\train.csv
Validation: ..\data\processed\validation.csv
Test: ..\data\processed\test.csv


In [14]:
# 7.11 — Final verification of saved split datasets

train_check = pd.read_csv(train_path, parse_dates=["date"])
validation_check = pd.read_csv(validation_path, parse_dates=["date"])
test_check = pd.read_csv(test_path, parse_dates=["date"])

print("TRAIN")
print("Shape:", train_check.shape)
print("Date:", train_check["date"].min(), "→", train_check["date"].max())

print("\nVALIDATION")
print("Shape:", validation_check.shape)
print("Date:", validation_check["date"].min(), "→", validation_check["date"].max())

print("\nTEST")
print("Shape:", test_check.shape)
print("Date:", test_check["date"].min(), "→", test_check["date"].max())

print("\nFINAL CHECKS")
print(
    "Row counts match:",
    len(train_check) == len(train_df)
    and len(validation_check) == len(validation_df)
    and len(test_check) == len(test_df)
)

print(
    "Columns match:",
    list(train_check.columns) == list(df.columns)
    and list(validation_check.columns) == list(df.columns)
    and list(test_check.columns) == list(df.columns)
)

print(
    "No missing values:",
    train_check.isnull().sum().sum() == 0
    and validation_check.isnull().sum().sum() == 0
    and test_check.isnull().sum().sum() == 0
)

TRAIN
Shape: (103272, 27)
Date: 2014-01-01 00:00:00 → 2015-05-25 00:00:00

VALIDATION
Shape: (63109, 27)
Date: 2015-05-26 00:00:00 → 2015-09-11 00:00:00

TEST
Shape: (81521, 27)
Date: 2015-09-12 00:00:00 → 2015-12-30 00:00:00

FINAL CHECKS
Row counts match: True
Columns match: True
No missing values: True
